# `param_shift` walkthrough

Build a small real torchsynth dataset, add the `shift` sensitivity column, and browse the
result in SmooSense.

`param_shift` assigns every row exactly one parameter of the synth's param spec, redraws
that parameter from its own distribution, re-renders the patch, and scores the perturbed
audio against the row's stored audio. The seven facets land in one nested `shift` struct:

| Subfield | Meaning |
| --- | --- |
| `shift.param` | Name of the one parameter shifted on this row |
| `shift.amount` | Size of the shift in encoded space (L2 over the parameter's span) |
| `shift.audio` | Audio rendered from the shifted patch |
| `shift.rms` | RMS-envelope cosine **similarity** — 1.0 means unchanged |
| `shift.sot` | Sliced-optimal-transport **distance** |
| `shift.wmfcc` | DTW-normalised MFCC **distance** |
| `shift.mss` | Multi-scale spectrogram **distance** |

Everything below is the real production path: the real writer, the real renderer, the real
`synth-setter-add-embeddings` CLI. Nothing is faked or mocked.

In [1]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

# The render loop's progress bar and its per-sample log lines are useful live but write
# hundreds of lines into a saved notebook. Both must be set before synth_setter is imported.
# Drop these two lines to watch the render progress interactively.
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOGURU_LEVEL"] = "WARNING"

import lance
import numpy as np

from synth_setter.data.vst.shapes import AUDIO_FIELD, SHIFT_FIELD
from synth_setter.data.vst.writers import make_lance_dataset
from synth_setter.param_spec_name import ParamSpecName
from synth_setter.pipeline.schemas.spec import RenderConfig
from synth_setter.synth_spec import SynthName, SynthSpec

ROWS = 50
SYNTH = "torchsynth_adsr"
SAMPLE_RATE = 22_050
DURATION_SECONDS = 0.5
CHANNELS = 2
VELOCITY = 100
SHIFT_SEED = 20260730

## 1. Render a 50-row torchsynth dataset

torchsynth renders in-process, so this needs no plugin bundle and no R2 credentials.
`make_lance_dataset` is the same writer the distributed pipeline calls per shard.

In [2]:
render_config = RenderConfig(
    synth=SynthSpec(
        name=SynthName(SYNTH),
        param_spec_name=ParamSpecName(SYNTH),
        plugin_path="torchsynth",
        plugin_state_path="",
        synth_version="1.0.2",
    ),
    renderer_backend="torchsynth",
    sample_rate=SAMPLE_RATE,
    channels=CHANNELS,
    velocity=VELOCITY,
    signal_duration_seconds=DURATION_SECONDS,
    min_loudness=-70.0,
    samples_per_render_batch=8,
    samples_per_shard=ROWS,
    base_seed=42,
    plugin_reload_cadence="once",
    gui_toggle_cadence="never",
)

workdir = Path(tempfile.mkdtemp(prefix="param-shift-walkthrough-"))
uri = workdir / "shard-000000.lance"
make_lance_dataset(uri, render_config)

dataset = lance.dataset(str(uri))
print(f"{dataset.count_rows()} rows -> {uri}")
print("columns:", dataset.schema.names)

/home/k-linux/synth-setter-param-shift/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/k-linux/synth-setter-param-shift/.venv/lib/python3.12/site-packages/torchsynth/synth.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/k-linux/synth-setter-param-shift/src/synth_setter/data/vst/renderers.py:639: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  torch.tensor([row], dtype=torch.float32),


[2026-07-30T20:25:44Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/param-shift-walkthrough-0yhawphj/.shard-000000.lance.tmp-tgx_76bv, it will be created


[2026-07-30T20:25:44Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/param-shift-walkthrough-0yhawphj/.shard-000000.lance.tmp-tgx_76bv, it will be created


[2026-07-30T20:25:45Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/param-shift-walkthrough-0yhawphj/.shard-000000.lance.tmp-tgx_76bv, it will be created


[2026-07-30T20:25:46Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/param-shift-walkthrough-0yhawphj/.shard-000000.lance.tmp-tgx_76bv, it will be created


[2026-07-30T20:25:47Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/param-shift-walkthrough-0yhawphj/.shard-000000.lance.tmp-tgx_76bv, it will be created


[2026-07-30T20:25:47Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/param-shift-walkthrough-0yhawphj/.shard-000000.lance.tmp-tgx_76bv, it will be created


50 rows -> /tmp/param-shift-walkthrough-0yhawphj/shard-000000.lance
columns: ['audio', 'mel_spec', 'param_array', 'debug', 'audio_mp3', 'audio_uuid']


[2026-07-30T20:25:48Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/param-shift-walkthrough-0yhawphj/.shard-000000.lance.tmp-tgx_76bv, it will be created


## 2. Add the `shift` column

The CLI runs as a real subprocess, exactly as an operator would invoke it. `render=` and
`synth=` compose the renderer the re-render goes through (#2565); the param spec comes from
that synth identity, so a shift can never be encoded against a spec the renderer does not
share.

`param_shift_seed` is namespaced away from the dataset's `base_seed`, so reusing the same
number here is harmless — the replacement draws come from a stream datagen never touches.

In [3]:
subprocess.run(  # noqa: S603 — sys.executable and every argument are notebook-owned
    [
        sys.executable,
        "-m",
        "synth_setter.pipeline.data.add_embeddings",
        f"lance_uri={uri}",
        "embeddings=[param_shift]",
        "render=torchsynth",
        f"synth={SYNTH}",
        f"render.sample_rate={SAMPLE_RATE}",
        f"render.channels={CHANNELS}",
        f"render.velocity={VELOCITY}",
        f"render.signal_duration_seconds={DURATION_SECONDS}",
        f"param_shift_seed={SHIFT_SEED}",
        "batch_size=10",
        "build_index=false",
    ],
    check=True,
)

dataset = lance.dataset(str(uri))
print(dataset.schema.field(SHIFT_FIELD))

2026-07-30 16:25:50 [info     ] lance_logging_configured       native_level=warn


2026-07-30 16:25:51 [info     ] adding_embeddings              batch_size=10 columns=['shift'] rows=50 sample_rate=22050 uri=/tmp/param-shift-walkthrough-0yhawphj/shard-000000.lance
2026-07-30 16:25:51 [info     ] loading_param_shift_renderer   backend=torchsynth param_spec=torchsynth_adsr seed=20260730 synth=torchsynth_adsr


/home/k-linux/synth-setter-param-shift/.venv/lib/python3.12/site-packages/torchsynth/synth.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/k-linux/synth-setter-param-shift/src/synth_setter/data/vst/renderers.py:639: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  torch.tensor([row], dtype=torch.float32),


2026-07-30 16:25:51 [info     ] inferring_embedding_schema     columns=['shift']


2026-07-30 16:25:52 [info     ] inferred_embedding_schema      columns=['shift']
2026-07-30 16:25:52 [info     ] embedding_write_started        batch_size=10 columns=['shift'] source_version=3 total_rows=50


...

2026-07-30 16:25:52 [info     ] embedding_progress             batch_ms=297.2 batch_rows=10 interbatch_ms=0.6 param_shift_ms=297.2 percent=40.0 rows_per_second=32.8 rows_processed=20 total_rows=50


2026-07-30 16:25:53 [info     ] embedding_progress             batch_ms=306.1 batch_rows=10 interbatch_ms=0.2 param_shift_ms=306.0 percent=60.0 rows_per_second=32.8 rows_processed=30 total_rows=50


2026-07-30 16:25:53 [info     ] embedding_progress             batch_ms=307.5 batch_rows=10 interbatch_ms=0.2 param_shift_ms=307.4 percent=80.0 rows_per_second=32.7 rows_processed=40 total_rows=50


2026-07-30 16:25:53 [info     ] embedding_progress             batch_ms=381.0 batch_rows=10 interbatch_ms=0.2 param_shift_ms=381.0 percent=100.0 rows_per_second=31.2 rows_processed=50 total_rows=50
2026-07-30 16:25:53 [info     ] wrote_embeddings               columns=['shift'] committed_version=4 rows_processed=50 total_rows=50
2026-07-30 16:25:53 [info     ] added_embeddings               columns=['shift'] uri=/tmp/param-shift-walkthrough-0yhawphj/shard-000000.lance


pyarrow.Field<shift: struct<param: string, amount: float, audio: extension<arrow.fixed_shape_tensor[value_type=halffloat, shape=[2,11025], permutation=[0,1]]>, rms: float, sot: float, wmfcc: float, mss: float>>


## 3. Read the shift back

Lance projects struct subfields directly, so a query for `shift.param` and `shift.mss`
never materialises the re-rendered audio.

In [4]:
scores = dataset.to_table(
    columns={
        "param": f"{SHIFT_FIELD}.param",
        "amount": f"{SHIFT_FIELD}.amount",
        "rms": f"{SHIFT_FIELD}.rms",
        "sot": f"{SHIFT_FIELD}.sot",
        "wmfcc": f"{SHIFT_FIELD}.wmfcc",
        "mss": f"{SHIFT_FIELD}.mss",
    }
).to_pandas()

scores.head(10)

,param,amount,rms,sot,wmfcc,mss
0,adsr_1.attack,0.479817,1.000000,0.000254,18.706219,3.441050
1,adsr_1.decay,0.198400,1.000000,0.000296,0.022960,0.004055
2,adsr_1.sustain,0.136302,1.000000,0.000122,0.016092,0.004165
3,adsr_1.release,0.613758,1.000000,0.000147,0.019967,0.005661
4,vco_2.shape,0.690247,0.999952,0.008059,5.555688,2.849673
5,pitch,0.291667,0.999718,0.006777,3.700065,2.625604
6,note_start_and_end,0.446847,0.207481,0.363461,15.520386,19.472059
7,adsr_1.attack,0.061681,1.000000,0.000904,2.627417,0.059772
8,adsr_1.decay,0.293909,1.000000,0.000039,0.009611,0.002542
9,adsr_1.sustain,0.140507,0.998658,0.003140,0.225036,0.361951


### Rows are spread evenly across the spec

Assignment keys on the Lance row id, so each parameter owns the same share of rows give or
take one — and the same row always draws the same replacement, including after a
resume-cache replay.

In [5]:
scores["param"].value_counts().sort_index()

param
adsr_1.attack         8
adsr_1.decay          7
adsr_1.release        7
adsr_1.sustain        7
note_start_and_end    7
pitch                 7
vco_2.shape           7
Name: count, dtype: int64

### Which parameters move the sound most

This is the question the column exists to answer: per parameter, how far does the audio
travel when only that knob is redrawn. `rms` is a similarity (lower = more changed); the
other three are distances (higher = more changed).

In [6]:
summary = (
    scores.groupby("param")[["amount", "rms", "sot", "wmfcc", "mss"]]
    .mean()
    .sort_values("mss", ascending=False)
)
summary

,amount,rms,sot,wmfcc,mss
param,,,,,
note_start_and_end,0.502339,0.129090,0.470248,22.515078,47.657036
pitch,0.440476,0.999932,0.011459,7.114872,4.433204
adsr_1.attack,0.150420,0.944161,0.000494,5.583589,2.116491
vco_2.shape,0.306667,0.999990,0.004274,2.165059,1.027121
adsr_1.sustain,0.296650,0.992034,0.000678,0.307746,0.355195
adsr_1.decay,0.222052,0.998641,0.000974,0.226234,0.152645
adsr_1.release,0.427347,1.000000,0.000092,0.013502,0.003601


This notebook keeps its executed outputs in git, so the table above is the real result
of the run recorded below — `base_seed=42`, `param_shift_seed=20260730`, 50 rows of
`torchsynth_adsr` at 22.05 kHz. It is repeated here as markdown so the ranking survives any
future output stripping and shows up in a diff:

| `shift.param` | `amount` | `rms` ↓ | `sot` ↑ | `wmfcc` ↑ | `mss` ↑ |
| --- | --- | --- | --- | --- | --- |
| `note_start_and_end` | 0.502 | 0.129 | 0.470 | 22.515 | 47.657 |
| `pitch` | 0.440 | 1.000 | 0.011 | 7.115 | 4.433 |
| `adsr_1.attack` | 0.150 | 0.944 | 0.000 | 5.584 | 2.116 |
| `vco_2.shape` | 0.307 | 1.000 | 0.004 | 2.165 | 1.027 |
| `adsr_1.sustain` | 0.297 | 0.992 | 0.001 | 0.308 | 0.355 |
| `adsr_1.decay` | 0.222 | 0.999 | 0.001 | 0.226 | 0.153 |
| `adsr_1.release` | 0.427 | 1.000 | 0.000 | 0.014 | 0.004 |

Note timing dominates, and `adsr_1.release` is inert — `rms` is exactly 1.0 and every distance
is ~0, because a 0.5 s render ends before the release stage shapes anything. Both readings are
the column working, not failing. Your numbers will differ with the seeds and the synth.

A zero `amount` is legitimate rather than a bug: `pitch` is discrete, so a redraw can land
back on its original value, and such a row should score as unchanged — `rms` at 1.0 and the
three distances at 0. At 50 rows this is usually empty (`pitch` owns about a seventh of the
rows and has 25 values, so roughly 0.3 rows are expected); the check is here because a
*non*-empty result whose scores were not ~unchanged would mean the recorded `amount` and the
recorded audio disagree.

In [7]:
unchanged = scores[scores["amount"] == 0.0]
print(f"{len(unchanged)} row(s) redrew their original value: {unchanged['param'].tolist()}")
unchanged

0 row(s) redrew their original value: []


,param,amount,rms,sot,wmfcc,mss


### The recorded audio really is the recorded patch

`shift.audio` is stored exactly like `audio` — same shape, same dtype — so the pair is
directly comparable.

In [8]:
audio = dataset.to_table(columns=[AUDIO_FIELD]).column(AUDIO_FIELD)
audio = audio.combine_chunks().to_numpy_ndarray()
shift_audio = (
    dataset.to_table(columns={"a": f"{SHIFT_FIELD}.audio"})
    .column("a")
    .combine_chunks()
    .to_numpy_ndarray()
)

print("audio      ", audio.shape, audio.dtype)
print("shift.audio", shift_audio.shape, shift_audio.dtype)

loudest = int(np.argmax(scores["mss"].to_numpy()))
print(
    f"\nrow {loudest} shifted {scores['param'].iloc[loudest]!r} "
    f"by {scores['amount'].iloc[loudest]:.4f} -> mss {scores['mss'].iloc[loudest]:.3f}"
)

audio       (50, 2, 11025) float16
shift.audio (50, 2, 11025) float16

row 20 shifted 'note_start_and_end' by 0.4369 -> mss 72.455


Listen to the biggest mover — the original patch, then the same patch with one parameter
redrawn.

In [9]:
from IPython.display import Audio, display

display(Audio(audio[loudest].astype(np.float32), rate=SAMPLE_RATE))
display(Audio(shift_audio[loudest].astype(np.float32), rate=SAMPLE_RATE))

/home/k-linux/.cache/uv/archive-v0/hv2HGv_TWtPTEI3Ykn__X/lib/python3.12/site-packages/IPython/lib/display.py:188: RuntimeWarning: invalid value encountered in divide
  scaled = data / normalization_factor * 32767
/home/k-linux/.cache/uv/archive-v0/hv2HGv_TWtPTEI3Ykn__X/lib/python3.12/site-packages/IPython/lib/display.py:189: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


## 4. Browse the scores with the `Sense` widget

`Sense` takes a DataFrame directly — the one-liner from
[SmooSense's notebook guide](https://smoosense.ai/blogs/jupyter-notebook/). It is the right
tool for the scalar scores because they hold no vector columns, so its DataFrame -> Parquet
round trip is lossless here.

Both this and the dataset browser below serve from the same local server, so neither renders
on GitHub — run the notebook to see them.

In [10]:
from smoosense.widget import Sense

Sense(scores)

 * Serving Flask app 'smoosense.app'


 * Debug mode: off


SmooSense server started at http://localhost:8001


## 5. Browse the whole dataset in SmooSense

For the full dataset, point SmooSense at the `.lance` directory instead of a DataFrame.
`shift.audio` is a fixed-shape tensor nested inside a struct; the DataFrame -> Parquet path
`Sense` uses would flatten those types, and it would copy 50 re-rendered clips through a
temporary file to do it. Reading the Lance table natively keeps the column types intact.

Sort by `shift.mss` to rank rows by how much the redraw moved the sound, or group by
`shift.param` to compare parameters against each other.

The frame below is served from `localhost`, so it only appears in a live kernel — on GitHub
this cell renders as code and nothing else.

> SmooSense is an optional in-notebook viewer, not a pipeline dependency; it is installed by
> the project `notebooks` dependency group with the `jupyter` extra
> ([#1681](https://github.com/tinaudio/synth-setter/issues/1681)). Run
> `uv sync --group notebooks` if the import fails.

In [11]:
from IPython.display import IFrame
from smoosense.widget import _SmooSenseServer

server = _SmooSenseServer()
server.start_if_needed()
IFrame(f"{server.base_url}/Table?tablePath={uri}", width="100%", height=900)